# ENTREGA FINAL

Blanca Gallardo, Ferran Tubert i Agustí Costabella - 15 de febrero de 2026

## Carga de librerías y datos

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
from h2o.display import display
from scipy.stats import f_oneway
import h2o
from h2o.grid.grid_search import H2OGridSearch
from h2o.estimators import H2OGradientBoostingEstimator, H2ORandomForestEstimator, H2OGeneralizedLinearEstimator

In [2]:
from src.MovieDatabase import MovieDatabase

mdb = MovieDatabase(uri="mongodb://localhost:27017/", db_name="tmdb_movies_db")

In [3]:
# Extraer todas las colecciones
movies_docs = list(mdb.movies.find({}))
genres_docs = list(mdb.genres.find({}))
production_companies_docs = list(mdb.production_companies.find({}))
production_countries_docs = list(mdb.production_countries.find({}))
language_docs = list(mdb.languages.find({}))

# Convertir a Polars
movies_df = pl.from_dicts(movies_docs).drop("_id")
genres_df = pl.from_dicts(genres_docs).drop("_id")
production_companies_df = pl.from_dicts(production_companies_docs).drop("_id")
production_countries_df = pl.from_dicts(production_countries_docs).drop("_id")
language_df = pl.from_dicts(language_docs).drop("_id")


In [4]:
movies_lazy = movies_df.lazy()
genres_lazy = genres_df.lazy()
companies_lazy = production_companies_df.lazy()
countries_lazy = production_countries_df.lazy()
languages_lazy = language_df.lazy()

In [5]:
# Movies
display(movies_lazy.select(pl.all()).limit(5).collect())

# Genres
display(genres_lazy.select(pl.all()).limit(5).collect())

# Production companies
display(companies_lazy.select(pl.all()).limit(5).collect())

# Production countries
display(countries_lazy.select(pl.all()).limit(5).collect())

# Languages
display(languages_lazy.select(pl.all()).limit(5).collect())

id,title,original_title,release_date,vote_average,vote_count,popularity,runtime,original_language,revenue,overview,tagline,adult,homepage,budget,genre_ids,company_ids,country_ids,language_ids
i64,str,str,str,f64,i64,f64,i64,str,i64,str,str,bool,str,i64,list[i64],list[i64],list[i64],list[i64]
27205,"""Inception""","""Inception""","""2010-07-15""",8.364,34495,83.952,148,"""en""",825532764,"""Cobb, a skilled thief who comm…","""Your mind is the scene of the …",false,"""https://www.warnerbros.com/mov…",160000000,"[6, 0, 15]","[2517, 8266, 9811]","[59, 17]","[61, 104, … 76]"
157336,"""Interstellar""","""Interstellar""","""2014-11-05""",8.417,32571,140.241,169,"""en""",701729206,"""The adventures of a group of e…","""Mankind was born on Earth. It …",false,"""http://www.interstellarmovie.n…",165000000,"[15, 17, 0]","[2517, 8266, 2707]","[59, 17]",[61]
155,"""The Dark Knight""","""The Dark Knight""","""2008-07-16""",8.512,30619,130.643,152,"""en""",1004558444,"""Batman raises the stakes in hi…","""Welcome to a world without rul…",false,"""https://www.warnerbros.com/mov…",185000000,"[17, 6, … 3]","[5032, 2517, … 9811]","[59, 17]","[61, 77]"
19995,"""Avatar""","""Avatar""","""2009-12-15""",7.573,29815,79.932,162,"""en""",2923706026,"""In the 22nd century, a paraple…","""Enter the world of Pandora.""",false,"""https://www.avatar.com/movies/…",237000000,"[6, 15, … 0]","[2180, 3249, … 9174]","[17, 59]","[61, 13]"
24428,"""The Avengers""","""The Avengers""","""2012-04-25""",7.71,29166,98.082,143,"""en""",1518815515,"""When an unexpected enemy emerg…","""Some assembly required.""",false,"""https://www.marvel.com/movies/…",220000000,"[0, 6, 15]",[8344],[17],"[61, 83, 33]"


genre_id,name
i64,str
0,"""Science Fiction"""
1,"""Family"""
2,"""Mystery"""
3,"""Thriller"""
4,"""Fantasy"""


company_id,name
i64,str
0,"""MACA Entretenimento"""
1,"""Les Films du Cru"""
2,"""BIM Distribuzione"""
3,"""Why Not Productions"""
4,"""Organism Pictures"""


country_id,name
i64,str
0,"""Guatemala"""
1,"""Spain"""
2,"""Chile"""
3,"""India"""
4,"""Yugoslavia"""


language_id,name
i64,str
0,"""German"""
1,"""Quechua"""
2,"""Ukrainian"""
3,""""""
4,"""Burmese"""


## Limpieza y preparación

In [6]:
# Movies
movies_cols = movies_lazy.collect_schema().names()
movies_nulls = movies_lazy.select([
    pl.col(c).is_null().sum().alias(f"{c}_nulls") for c in movies_cols
]).collect()
display(movies_nulls)

# Genres
genres_cols = genres_lazy.collect_schema().names()
genres_nulls = genres_lazy.select([
    pl.col(c).is_null().sum().alias(f"{c}_nulls") for c in genres_cols
]).collect()
display(genres_nulls)

# Production Companies
companies_cols = companies_lazy.collect_schema().names()
companies_nulls = companies_lazy.select([
    pl.col(c).is_null().sum().alias(f"{c}_nulls") for c in companies_cols
]).collect()
display(companies_nulls)

# Production Countries
countries_cols = countries_lazy.collect_schema().names()
countries_nulls = countries_lazy.select([
    pl.col(c).is_null().sum().alias(f"{c}_nulls") for c in countries_cols
]).collect()
display(countries_nulls)

# Languages
languages_cols = languages_lazy.collect_schema().names()
languages_nulls = languages_lazy.select([
    pl.col(c).is_null().sum().alias(f"{c}_nulls") for c in languages_cols
]).collect()
display(languages_nulls)

id_nulls,title_nulls,original_title_nulls,release_date_nulls,vote_average_nulls,vote_count_nulls,popularity_nulls,runtime_nulls,original_language_nulls,revenue_nulls,overview_nulls,tagline_nulls,adult_nulls,homepage_nulls,budget_nulls,genre_ids_nulls,company_ids_nulls,country_ids_nulls,language_ids_nulls
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


genre_id_nulls,name_nulls
u32,u32
0,0


company_id_nulls,name_nulls
u32,u32
0,0


country_id_nulls,name_nulls
u32,u32
0,0


language_id_nulls,name_nulls
u32,u32
0,0


In [7]:
# Movies
movies_duplicates = movies_lazy.select([
    pl.col("id").is_unique().alias("id_unique")
]).collect()
num_movies_duplicates = (movies_duplicates["id_unique"] == False).sum()
print(f"Duplicados en movies: {num_movies_duplicates}")

# Genres
genres_duplicates = genres_lazy.select([
    pl.col("genre_id").is_unique().alias("id_unique")
]).collect()
num_genres_duplicates = (genres_duplicates["id_unique"] == False).sum()
print(f"Duplicados en genres: {num_genres_duplicates}")

# Production Companies
companies_duplicates = companies_lazy.select([
    pl.col("company_id").is_unique().alias("id_unique")
]).collect()
num_companies_duplicates = (companies_duplicates["id_unique"] == False).sum()
print(f"Duplicados en companies: {num_companies_duplicates}")

# Production Countries
countries_duplicates = countries_lazy.select([
    pl.col("country_id").is_unique().alias("id_unique")
]).collect()
num_countries_duplicates = (countries_duplicates["id_unique"] == False).sum()
print(f"Duplicados en countries: {num_countries_duplicates}")

# Languages
languages_duplicates = languages_lazy.select([
    pl.col("language_id").is_unique().alias("id_unique")
]).collect()
num_languages_duplicates = (languages_duplicates["id_unique"] == False).sum()
print(f"Duplicados en languages: {num_languages_duplicates}")

Duplicados en movies: 0
Duplicados en genres: 0
Duplicados en companies: 0
Duplicados en countries: 0
Duplicados en languages: 0


In [15]:
import polars as pl

movies_df_final = (
    movies_lazy
    .with_columns([
        # Convertir release_date a Date
        pl.col("release_date").str.strptime(pl.Date, "%Y-%m-%d").alias("release_date"),
        # adult -> is_adult 0/1
        pl.col("adult").cast(pl.Int8).alias("is_adult")
    ])
    .with_columns([
        # Extraer año y mes desde la columna release_date ya convertida
        pl.col("release_date").dt.year().alias("release_year"),
        pl.col("release_date").dt.month().alias("release_month")
    ])
    .select([
        "id",
        "vote_average",
        "vote_count",
        "release_year",
        "release_month",
        "revenue",
        "runtime",
        "is_adult",
        "budget",
        "original_language",
        "popularity",
        pl.col("genre_ids").alias("id_genre"),
        pl.col("language_ids").alias("id_languages"),
        pl.col("company_ids").alias("id_production"),
        pl.col("country_ids").alias("id_country")
    ])
)

# Ejecutar LazyFrame y mostrar las primeras 5 filas
display(movies_df_final.collect().head(5))

id,vote_average,vote_count,release_year,release_month,revenue,runtime,is_adult,budget,original_language,popularity,id_genre,id_languages,id_production,id_country
i64,f64,i64,i32,i8,i64,i64,i8,i64,str,f64,list[i64],list[i64],list[i64],list[i64]
27205,8.364,34495,2010,7,825532764,148,0,160000000,"""en""",83.952,"[6, 0, 15]","[61, 104, … 76]","[2517, 8266, 9811]","[59, 17]"
157336,8.417,32571,2014,11,701729206,169,0,165000000,"""en""",140.241,"[15, 17, 0]",[61],"[2517, 8266, 2707]","[59, 17]"
155,8.512,30619,2008,7,1004558444,152,0,185000000,"""en""",130.643,"[17, 6, … 3]","[61, 77]","[5032, 2517, … 9811]","[59, 17]"
19995,7.573,29815,2009,12,2923706026,162,0,237000000,"""en""",79.932,"[6, 15, … 0]","[61, 13]","[2180, 3249, … 9174]","[17, 59]"
24428,7.71,29166,2012,4,1518815515,143,0,220000000,"""en""",98.082,"[0, 6, 15]","[61, 83, 33]",[8344],[17]


## EDA

In [9]:
# Movies
display(movies_lazy.collect_schema())

# Genres
display(genres_lazy.collect_schema())

# Companies
display(companies_lazy.collect_schema())

# Countries
display(countries_lazy.collect_schema())

Schema([('id', Int64),
        ('title', String),
        ('original_title', String),
        ('release_date', String),
        ('vote_average', Float64),
        ('vote_count', Int64),
        ('popularity', Float64),
        ('runtime', Int64),
        ('original_language', String),
        ('revenue', Int64),
        ('overview', String),
        ('tagline', String),
        ('adult', Boolean),
        ('homepage', String),
        ('budget', Int64),
        ('genre_ids', List(Int64)),
        ('company_ids', List(Int64)),
        ('country_ids', List(Int64)),
        ('language_ids', List(Int64))])

Schema([('genre_id', Int64), ('name', String)])

Schema([('company_id', Int64), ('name', String)])

Schema([('country_id', Int64), ('name', String)])

## Feature Engineering

## Modelo

## Optimización

## Evaluación